# 2. Python-аналитика

Быстрые ответы в самом низу

# Python-1. Очистка данных и базовая витрина по взысканию

## Задание

#### Контекст

Нужно быстро собрать простую аналитическую витрину для еженедельного контроля результатов взыскания.

#### Даны файлы

`cases.csv`

* `case_id`
* `account_id`
* `customer_id`
* `start_date`
* `dpd_bucket`
* `strategy_name`
* `principal_amount`

`contacts.csv`

* `event_id`
* `case_id`
* `event_dttm`
* `channel`
* `contact_result`

`payments.csv`

* `payment_id`
* `account_id`
* `payment_dttm`
* `payment_amount`

#### Бизнес-правила

Нужно написать функцию `build_collection_summary(cases, contacts, payments)`, которая:

* удаляет дубли в `contacts` по `event_id`, оставляя запись с максимальным `event_dttm`;
* приводит даты к типу datetime;
* исключает из `payments` строки с `payment_amount <= 0`;
* считает по каждому `dpd_bucket`:
  * `total_cases`
* `contacted_cases_7d` - хотя бы один контакт в окне от `start_date` до `start_date + 6 дней`
* `paid_cases_7d` - хотя бы один платеж в окне от `start_date` до `start_date + 6 дней`
  * `collected_amount_7d`
  * `avg_payment_per_paid_case_7d`

#### Что нужно получить

Функцию, которая возвращает `DataFrame`, отсортированный по бакетам в порядке:

* `1-30`
* `31-60`
* `61-90`
* `90+`

#### Что прислать

Код функции и короткий пример вызова.

## Решение

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cases = pd.read_csv('cases.csv')
contacts = pd.read_csv('contacts.csv')
payments = pd.read_csv('payments.csv')

### Чтение

In [ ]:
cases.head()

,case_id,account_id,customer_id,start_date,dpd_bucket,strategy_name,principal_amount
0,1,50000,80000,2026-02-27,31-60,late_stage,68845.38
1,2,50001,80001,2026-02-18,1-30,late_stage,55274.70
2,3,50002,80002,2026-02-02,31-60,mixed,52909.61
3,4,50003,80003,2026-03-25,61-90,sms_first,64609.93
4,5,50004,80004,2026-01-24,1-30,sms_first,53838.74


In [ ]:
contacts.head()

,event_id,case_id,event_dttm,channel,contact_result
0,4505,1703,2026-03-15 17:11:00,sms,RPC
1,2588,983,2026-02-24 22:54:00,sms,Promise
2,6466,2419,2026-04-01 13:54:00,call,Promise
3,5671,2118,2026-02-08 18:29:00,call,No answer
4,4414,1669,2026-01-11 15:57:00,call,No answer


In [ ]:
payments.head()

,payment_id,account_id,payment_dttm,payment_amount
0,1,50002,2026-02-10 14:54:00,6397.25
1,2,50005,2026-02-03 16:03:00,25241.16
2,3,50006,2026-01-31 15:09:00,17364.78
3,4,50007,2026-02-04 08:36:00,13084.38
4,5,50008,2026-03-03 12:33:00,0.00


In [ ]:
cases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   case_id           2600 non-null   int64  
 1   account_id        2600 non-null   int64  
 2   customer_id       2600 non-null   int64  
 3   start_date        2600 non-null   object 
 4   dpd_bucket        2600 non-null   object 
 5   strategy_name     2600 non-null   object 
 6   principal_amount  2600 non-null   float64
dtypes: float64(1), int64(3), object(3)
memory usage: 142.3+ KB


In [ ]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7167 entries, 0 to 7166
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   event_id        7167 non-null   int64 
 1   case_id         7167 non-null   int64 
 2   event_dttm      7167 non-null   object
 3   channel         7167 non-null   object
 4   contact_result  7167 non-null   object
dtypes: int64(2), object(3)
memory usage: 280.1+ KB


In [ ]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1859 entries, 0 to 1858
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   payment_id      1859 non-null   int64  
 1   account_id      1859 non-null   int64  
 2   payment_dttm    1859 non-null   object 
 3   payment_amount  1859 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 58.2+ KB


### Перевод к datetime

In [ ]:
cases['start_date'] = pd.to_datetime(cases['start_date'])
contacts['event_dttm'] = pd.to_datetime(contacts['event_dttm'])
payments['payment_dttm'] = pd.to_datetime(payments['payment_dttm'])

In [ ]:
cases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   case_id           2600 non-null   int64         
 1   account_id        2600 non-null   int64         
 2   customer_id       2600 non-null   int64         
 3   start_date        2600 non-null   datetime64[ns]
 4   dpd_bucket        2600 non-null   object        
 5   strategy_name     2600 non-null   object        
 6   principal_amount  2600 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 142.3+ KB


### Очистка дублей `event_id` в `contacts`

In [ ]:
contacts = contacts.sort_values('event_dttm').drop_duplicates(subset='event_id', keep='last')

In [ ]:
contacts['event_id'].nunique(), len(contacts)

(6959, 6959)

### Очистка `payments`

In [ ]:
payments = payments[payments['payment_amount'] > 0]

In [ ]:
payments['payment_amount'].describe()

,payment_amount
count,1533.000000
mean,9909.105140
std,8448.287915
min,473.710000
25%,4715.960000
50%,8066.200000
75%,12523.600000
max,103590.110000


### Summary DF — Фичи по каждому `dpd_bucket`

`total_cases`

In [ ]:
total_cases = (cases
               .groupby('dpd_bucket', as_index=False)
               .agg(total_cases=('case_id', 'nunique')))

`contacted_cases_7d`

In [ ]:
# contacts к cases (join)
contacts_cases = contacts.merge(
    cases[['case_id', 'start_date', 'dpd_bucket']],
    on='case_id',
    how='inner'
)

# Фильтруем по окну 7 дней
contacts_7d = contacts_cases[
    (contacts_cases['event_dttm'] >= contacts_cases['start_date']) &
    (contacts_cases['event_dttm'] <= contacts_cases['start_date'] + pd.Timedelta(days=6))
]

# Уникальные кейсы (хотя бы один контакт)
contacted_cases_7d = (
    contacts_7d[['dpd_bucket', 'case_id']]
    .drop_duplicates()
    .groupby('dpd_bucket', as_index=False)
    .agg(contacted_cases_7d=('case_id', 'nunique'))
)

In [ ]:
contacts_7d.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2207 entries, 0 to 6584
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   event_id        2207 non-null   int64         
 1   case_id         2207 non-null   int64         
 2   event_dttm      2207 non-null   datetime64[ns]
 3   channel         2207 non-null   object        
 4   contact_result  2207 non-null   object        
 5   start_date      2207 non-null   datetime64[ns]
 6   dpd_bucket      2207 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(3)
memory usage: 137.9+ KB


`paid_cases_7d`

In [ ]:
# payments к cases (join)
payments_cases = payments.merge(
    cases[['case_id', 'account_id', 'start_date', 'dpd_bucket']],
    on='account_id',
    how='inner'
)

# Фильтруем по окну 7 дней
payments_7d = payments_cases[
    (payments_cases['payment_dttm'] >= payments_cases['start_date']) &
    (payments_cases['payment_dttm'] <= payments_cases['start_date'] + pd.Timedelta(days=6))
]

# Уникальные кейсы (хотя бы один платеж)
paid_cases_7d = (
    payments_7d[['dpd_bucket', 'case_id']]
    .drop_duplicates()
    .groupby('dpd_bucket', as_index=False)
    .agg(paid_cases_7d=('case_id', 'nunique'))
)

In [ ]:
paid_cases_7d.head()

,dpd_bucket,paid_cases_7d
0,1-30,323
1,31-60,170
2,61-90,90
3,90+,20


`collected_amount_7d`

In [ ]:
# Сумма платежей в окне 7 дней по каждому dpd_bucket
collected_amount_7d = (
    payments_7d
    .groupby('dpd_bucket', as_index=False)
    .agg(collected_amount_7d=('payment_amount', 'sum'))
)

In [ ]:
collected_amount_7d.head()

,dpd_bucket,collected_amount_7d
0,1-30,3367920.36
1,31-60,2349000.61
2,61-90,1317981.54
3,90+,345482.94


Финальный фрейм с фичами - `summary`

In [ ]:
# Соединяем фичи к summary
summary = total_cases
summary = summary.merge(contacted_cases_7d, on='dpd_bucket', how='left')
summary = summary.merge(paid_cases_7d, on='dpd_bucket', how='left')
summary = summary.merge(collected_amount_7d, on='dpd_bucket', how='left')

In [ ]:
summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   dpd_bucket           4 non-null      object 
 1   total_cases          4 non-null      int64  
 2   contacted_cases_7d   4 non-null      int64  
 3   paid_cases_7d        4 non-null      int64  
 4   collected_amount_7d  4 non-null      float64
dtypes: float64(1), int64(3), object(1)
memory usage: 292.0+ bytes


`avg_payment_per_paid_case_7d`

In [ ]:
summary['avg_payment_per_paid_case_7d'] = (
    summary['collected_amount_7d'] / summary['paid_cases_7d']
)

In [ ]:
summary.head()

,dpd_bucket,total_cases,contacted_cases_7d,paid_cases_7d,collected_amount_7d,avg_payment_per_paid_case_7d
0,1-30,1086,492,323,3367920.36,10426.998019
1,31-60,755,419,170,2349000.61,13817.650647
2,61-90,489,340,90,1317981.54,14644.239333
3,90+,270,196,20,345482.94,17274.147000


In [ ]:
summary.sort_values('dpd_bucket').reset_index(drop=True).head()

,dpd_bucket,total_cases,contacted_cases_7d,paid_cases_7d,collected_amount_7d,avg_payment_per_paid_case_7d
0,1-30,1086,492,323,3367920.36,10426.998019
1,31-60,755,419,170,2349000.61,13817.650647
2,61-90,489,340,90,1317981.54,14644.239333
3,90+,270,196,20,345482.94,17274.147000


Сортировка(хоть он и отсортирован)

In [ ]:
bucket_order = ['1-30', '31-60', '61-90', '90+']

summary['dpd_bucket'] = pd.Categorical(
    summary['dpd_bucket'],
    categories=bucket_order,
    ordered=True
)

summary = summary.sort_values('dpd_bucket').reset_index(drop=True)

### Функции

In [ ]:
def preprocess_data(cases: pd.DataFrame, contacts: pd.DataFrame, payments: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
  """Приводит даты к datetime, чистит дубли в contacts и убирает неположительные платежи."""
  cases['start_date'] = pd.to_datetime(cases['start_date'])
  contacts['event_dttm'] = pd.to_datetime(contacts['event_dttm'])
  payments['payment_dttm'] = pd.to_datetime(payments['payment_dttm'])

  contacts = contacts.sort_values('event_dttm').drop_duplicates(subset='event_id', keep='last')
  payments = payments[payments['payment_amount'] > 0]

  return cases, contacts, payments

In [ ]:
def filter_by_window(df: pd.DataFrame, event_col: str, start_col: str, days: int = 6) -> pd.DataFrame:
  """Выделяет события в окне [start_date, start_date + days]."""
  return df[(df[event_col] >= df[start_col]) & (df[event_col] <= df[start_col] + pd.Timedelta(days=days))]

In [ ]:
def make_collection_summary(cases: pd.DataFrame, contacts: pd.DataFrame, payments: pd.DataFrame, days: int = 6) -> pd.DataFrame:
  """Считает итоговую витрину по взысканию."""

  # total_cases
  summary = (cases
             .groupby('dpd_bucket', as_index=False)
             .agg(total_cases=('case_id', 'nunique'))
             .copy())

  # contacted_cases_7d
  contacts_cases = contacts.merge(
      cases[['case_id', 'start_date', 'dpd_bucket']],
      on='case_id',
      how='inner')

  contacts_window = filter_by_window(
      df=contacts_cases,
      event_col='event_dttm',
      start_col='start_date',
      days=days)

  contacted_cases = (contacts_window[['dpd_bucket', 'case_id']]
      .drop_duplicates()
      .groupby('dpd_bucket', as_index=False)
      .agg(contacted_cases_7d=('case_id', 'nunique')))

  # paid_cases_7d
  payments_cases = payments.merge(
      cases[['case_id', 'account_id', 'start_date', 'dpd_bucket']],
      on='account_id',
      how='inner')

  payments_window = filter_by_window(
      df=payments_cases,
      event_col='payment_dttm',
      start_col='start_date',
      days=days)

  paid_cases = (payments_window[['dpd_bucket', 'case_id']]
      .drop_duplicates()
      .groupby('dpd_bucket', as_index=False)
      .agg(paid_cases_7d=('case_id', 'nunique')))

  # collected_amount_7d
  collected_amount = (payments_window
      .groupby('dpd_bucket', as_index=False)
      .agg(collected_amount_7d=('payment_amount', 'sum')))

  # merge фичей в summary
  summary = summary.merge(contacted_cases, on='dpd_bucket', how='left')
  summary = summary.merge(paid_cases, on='dpd_bucket', how='left')
  summary = summary.merge(collected_amount, on='dpd_bucket', how='left')

  # заполняем пропуски
  metric_cols = ['contacted_cases_7d', 'paid_cases_7d', 'collected_amount_7d']
  summary[metric_cols] = summary[metric_cols].fillna(0)

  # avg_payment_per_paid_case_7d
  summary['avg_payment_per_paid_case_7d'] = (summary['collected_amount_7d'] / summary['paid_cases_7d'])
  summary['avg_payment_per_paid_case_7d'] = (summary['avg_payment_per_paid_case_7d'].replace([np.inf, -np.inf], 0).fillna(0))

  # сортировка бакетов
  bucket_order = ['1-30', '31-60', '61-90', '90+']
  summary['dpd_bucket'] = pd.Categorical(
      summary['dpd_bucket'],
      categories=bucket_order,
      ordered=True)
  summary = summary.sort_values('dpd_bucket').reset_index(drop=True)

  return summary

In [ ]:
def build_collection_summary(cases: pd.DataFrame, contacts: pd.DataFrame, payments: pd.DataFrame) -> pd.DataFrame:
  """Функция построения витрины."""
  cases = cases.copy()
  contacts = contacts.copy()
  payments = payments.copy()

  # --- проверки пустоты (базовые входные данные)
  if cases.empty:
      raise ValueError("cases is empty")

  if contacts.empty:
      print("Warning: contacts is empty")

  if payments.empty:
      print("Warning: payments is empty")

  cases, contacts, payments = preprocess_data(cases, contacts, payments)
  if payments.empty:
      print("Warning: no valid payments after filtering")

  summary = make_collection_summary(cases, contacts, payments, days=6)

  return summary

**Вызов функции:**

In [ ]:
summary = build_collection_summary(cases, contacts, payments)

In [ ]:
summary

,dpd_bucket,total_cases,contacted_cases_7d,paid_cases_7d,collected_amount_7d,avg_payment_per_paid_case_7d
0,1-30,1086,492,323,3367920.36,10426.998019
1,31-60,755,419,170,2349000.61,13817.650647
2,61-90,489,340,90,1317981.54,14644.239333
3,90+,270,196,20,345482.94,17274.147000


# Python-2. Анализ A/B-теста по SMS-напоминаниям

## Задание

#### Контекст

Команда протестировала новый текст SMS-напоминания. Нужно понять, дал ли он эффект.

#### Дан файл

`sms_ab_test.csv`

* `customer_id`
* `group_name` - `control` / `treatment`
* `segment` - сегмент клиента
* `delivered_flg` - SMS доставлено или нет (`0/1`)
* `paid_7d_flg` - была ли оплата в течение 7 дней (`0/1`)
* `paid_amount_7d` - сумма оплаты в течение 7 дней

#### Что нужно сделать

Напишите код, который:

* считает по каждому `segment` и в целом:
  * количество клиентов;
  * долю доставленных SMS;
  * `payment_rate_7d`;
  * `avg_paid_amount_per_customer_7d`;
* считает uplift `treatment` относительно `control` по `payment_rate_7d`;
* для общего `payment_rate_7d` проводит проверку статистической значимости разницы между `control` и `treatment`;
* формирует короткий вывод на 3-5 предложений.

#### Ограничения

* При расчете `payment_rate_7d` используйте всех клиентов из выборки, а не только доставленные SMS.
* Если используете статистический тест, явно напишите, почему выбрали именно его.

#### Что прислать

Код, итоговые таблицы и краткий вывод.

## Решение

### Постановка A/B-теста

В эксперименте сравниваются две группы:
- `control` (группа A) — пользователи с текущим текстом SMS;
- `treatment` (группа B) — пользователи с новым текстом SMS.

**Целевая метрика:** `payment_rate_7d` — доля клиентов, совершивших оплату в течение 7 дней.

1. **Нулевая гипотеза (H0):** разницы в `payment_rate_7d` между группами нет.  
2. **Альтернативная гипотеза (H1):** разница в `payment_rate_7d` между группами есть.

Для проверки статистической значимости будет использован `z-тест` для сравнения долей.

### Импорт и загрузка

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
df = pd.read_csv('sms_ab_test.csv')

### Просмотр фрейма

In [ ]:
df.head()

,customer_id,group_name,segment,delivered_flg,paid_7d_flg,paid_amount_7d
0,300000,control,high_risk,1,0,0.0
1,300001,treatment,medium_risk,1,0,0.0
2,300002,control,medium_risk,1,0,0.0
3,300003,treatment,high_risk,1,0,0.0
4,300004,control,medium_risk,1,0,0.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     7200 non-null   int64  
 1   group_name      7200 non-null   object 
 2   segment         7200 non-null   object 
 3   delivered_flg   7200 non-null   int64  
 4   paid_7d_flg     7200 non-null   int64  
 5   paid_amount_7d  7200 non-null   float64
dtypes: float64(1), int64(3), object(2)
memory usage: 337.6+ KB


In [ ]:
df.shape

(7200, 6)

In [ ]:
df['group_name'].value_counts(dropna=False) # баланс

,count
group_name,
control,3600
treatment,3600


Группы **сбалансированы**

In [ ]:
df['segment'].value_counts(dropna=False) # распределение сегментов

,count
segment,
medium_risk,2620
low_risk,1974
high_risk,1717
newcomers,889


In [ ]:
df[['delivered_flg', 'paid_7d_flg']].apply(pd.Series.value_counts, dropna=False)

,delivered_flg,paid_7d_flg
0,508,5516
1,6692,1684


### Метрики по segment и group_name

In [ ]:
segment_metrics = (df
    .groupby(['segment', 'group_name'], as_index=False)
    .agg(customers_cnt=('customer_id', 'nunique'),
         delivered_rate=('delivered_flg', 'mean'),
         payment_rate_7d=('paid_7d_flg', 'mean'),
         avg_paid_amount_per_customer_7d=('paid_amount_7d', 'mean')))

In [ ]:
segment_metrics

,segment,group_name,customers_cnt,delivered_rate,payment_rate_7d,avg_paid_amount_per_customer_7d
0,high_risk,control,863,0.900348,0.123986,1234.474762
1,high_risk,treatment,854,0.879391,0.148712,1424.542541
2,low_risk,control,987,0.959473,0.313070,1763.592665
3,low_risk,treatment,987,0.954407,0.372847,2083.512199
4,medium_risk,control,1296,0.932099,0.221451,1623.122955
5,medium_risk,treatment,1324,0.939577,0.237915,1767.628595
6,newcomers,control,454,0.916300,0.180617,1191.874802
7,newcomers,treatment,435,0.935632,0.204598,1296.637839


### Метрики "в целом"

In [ ]:
overall_metrics = (df
    .groupby('group_name', as_index=False)
    .agg(
        customers_cnt=('customer_id', 'nunique'),
        delivered_rate=('delivered_flg', 'mean'),
        payment_rate_7d=('paid_7d_flg', 'mean'),
        avg_paid_amount_per_customer_7d=('paid_amount_7d', 'mean')))

In [ ]:
overall_metrics

,group_name,customers_cnt,delivered_rate,payment_rate_7d,avg_paid_amount_per_customer_7d
0,control,3600,0.930000,0.218056,1514.082275
1,treatment,3600,0.928889,0.249722,1715.934331


### Uplift(эффект)

In [ ]:
control_rate = overall_metrics.loc[overall_metrics['group_name'] == 'control', 'payment_rate_7d'].values[0]

treatment_rate = overall_metrics.loc[overall_metrics['group_name'] == 'treatment', 'payment_rate_7d'].values[0]

In [ ]:
uplift = treatment_rate - control_rate

In [ ]:
uplift

np.float64(0.031666666666666676)

**Наблюдение:** `+3.17` процентных пункта к оплате — положительный эффект `treatment` группы

In [ ]:
# относительный (Заметка)
# rel_uplift = uplift / control_rate
# относительно baseline

### Статистический тест (проверка значимости)

Для проверки статистической значимости разницы в `payment_rate_7d` использован `z-test` для сравнения долей, так как **метрика является бинарной** (paid_7d_flg: 0/1), а группы control и treatment независимы.

**Дополнительно проведён** `chi-test` для таблицы сопряжённости 2x2 как альтернативный метод проверки. Оба теста подходят для анализа различий долей и при достаточном размере выборки дают согласованные результаты. Chi-test тоже подходит **для бинарной метрики**.

- `z-test` напрямую сравнивает доли между двумя группами
- `chi-test` проверяет зависимость между группой и фактом оплаты в таблице сопряжённости.

In [ ]:
# количество успехов (оплат)
success_control = df.loc[df['group_name'] == 'control', 'paid_7d_flg'].sum()
success_treatment = df.loc[df['group_name'] == 'treatment', 'paid_7d_flg'].sum()

# количество наблюдений
n_control = df.loc[df['group_name'] == 'control', 'paid_7d_flg'].count()
n_treatment = df.loc[df['group_name'] == 'treatment', 'paid_7d_flg'].count()

**Z-test**

In [ ]:
# доли
p_control = success_control / n_control
p_treatment = success_treatment / n_treatment

# объединённая оценка (pooled)
p_pool = (success_control + success_treatment) / (n_control + n_treatment)

# стандартная ошибка
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))

# z-статистика
z_stat = (p_treatment - p_control) / se

# p-value (двусторонний тест)
from scipy.stats import norm
p_value_z = 2 * (1 - norm.cdf(abs(z_stat)))

In [ ]:
z_stat, p_value_z

(np.float64(3.1738611784163986), np.float64(0.0015042561770188811))

**Chi-test**

In [ ]:
from scipy.stats import chi2_contingency

# таблица сопряжённости
contingency_table = [[success_control, n_control - success_control],
                     [success_treatment, n_treatment - success_treatment]]

chi2_stat, p_value_chi, dof, expected = chi2_contingency(contingency_table)

In [ ]:
chi2_stat, p_value_chi

(np.float64(9.897443670669132), np.float64(0.0016550852275749991))

In [ ]:
print("Z-test p-value:", p_value_z)
print("Chi-test p-value:", p_value_chi)

Z-test p-value: 0.0015042561770188811
Chi-test p-value: 0.0016550852275749991


- Вывод по обоим тестам совпадает
- `0.0015 << 0.05` — **эффект является статистически значимым**

### Вывод

Эффект является не только статистически значимым, но и практически заметным:
- В группе `treatment` наблюдается **более высокий** `payment_rate_7d` по сравнению с `control` — **(+3.17 п.п.)**, что указывает на **положительный эффект нового текста SMS**.
- Результаты статистических тестов (`z-test` и `chi-test`) показывают, что **различие является статистически значимым** (`p-value < 0.05`), что позволяет отвергнуть нулевую гипотезу об отсутствии эффекта.
- Таким образом, можно сделать вывод, что **новый текст SMS действительно увеличивает вероятность оплаты клиентов в течение 7 дней**.
- **Дополнительно** наблюдается рост среднего платежа на клиента, что усиливает практическую значимость результата.

# Python-3. Построение дневной витрины признаков



## Задание

#### Контекст

Перед обучением модели нужно из событийной истории построить витрину признаков без утечки в будущее.

#### Даны файлы

`actions_log.csv`

* `customer_id`
* `action_dttm`
* `action_type` - `call`, `sms`, `promise`, `payment`
* `action_amount` - сумма действия, для `payment` и `promise`, иначе `0`

`daily_snapshot.csv`

* `customer_id`
* `snapshot_date`
* `dpd`
* `outstanding_amount`

#### Что нужно сделать

Для каждой строки из `daily_snapshot.csv` постройте признаки, используя только события строго раньше `snapshot_date`:

* `calls_last_7d`
* `sms_last_7d`
* `promises_last_30d`
* `payments_amount_last_30d`
* `days_since_last_payment`
* `promised_amount_last_30d`

После этого:

* соберите итоговую витрину в один `DataFrame`;
* найдите топ-20 клиентов с максимальным числом контактов (`calls_last_7d + sms_last_7d`) и нулевой суммой оплат за последние 30 дней.

#### Важное условие

Нельзя использовать события на `snapshot_date` и позже.

#### Что прислать

Код, итоговую схему витрины и пример выборки топ-20 клиентов.

## Решение

### Импорт и загрузка

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
actions = pd.read_csv('actions_log.csv')
snapshot = pd.read_csv('daily_snapshot.csv')

### Quick Look

In [ ]:
actions.head()

,customer_id,action_dttm,action_type,action_amount
0,400000,2026-01-01 14:59:00,payment,13309.31
1,400000,2026-01-01 19:11:00,call,0.00
2,400000,2026-01-08 14:37:00,payment,6772.92
3,400000,2026-01-10 08:04:00,call,0.00
4,400000,2026-01-12 14:06:00,payment,8731.15


In [ ]:
snapshot.head()

,customer_id,snapshot_date,dpd,outstanding_amount
0,400000,2026-03-11,10,19000.14
1,400000,2026-03-15,22,16914.99
2,400001,2026-03-30,26,16142.60
3,400001,2026-04-03,33,16326.84
4,400002,2026-02-21,90,44636.91


In [ ]:
actions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14954 entries, 0 to 14953
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    14954 non-null  int64  
 1   action_dttm    14954 non-null  object 
 2   action_type    14954 non-null  object 
 3   action_amount  14954 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 467.4+ KB


In [ ]:
snapshot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         4500 non-null   int64  
 1   snapshot_date       4500 non-null   object 
 2   dpd                 4500 non-null   int64  
 3   outstanding_amount  4500 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 140.8+ KB


In [ ]:
actions['action_type'].value_counts(dropna=False)

,count
action_type,
call,6220
sms,4341
payment,2284
promise,2109


In [ ]:
snapshot['customer_id'].nunique(), len(snapshot)

(1500, 4500)

### Приведение дат

In [ ]:
actions['action_dttm'] = pd.to_datetime(actions['action_dttm'])
snapshot['snapshot_date'] = pd.to_datetime(snapshot['snapshot_date'])

In [ ]:
actions.info()
print('-'*50)
snapshot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14954 entries, 0 to 14953
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    14954 non-null  int64         
 1   action_dttm    14954 non-null  datetime64[ns]
 2   action_type    14954 non-null  object        
 3   action_amount  14954 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 467.4+ KB
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         4500 non-null   int64         
 1   snapshot_date       4500 non-null   datetime64[ns]
 2   dpd                 4500 non-null   int64         
 3   outstanding_amount  4500 non-null   float64       
dtypes: da

### Merge snapshot + actions

In [ ]:
snapshot_actions = snapshot.merge(actions,
    on='customer_id',
    how='left')

In [ ]:
snapshot_actions.head()

,customer_id,snapshot_date,dpd,outstanding_amount,action_dttm,action_type,action_amount
0,400000,2026-03-11,10,19000.14,2026-01-01 14:59:00,payment,13309.31
1,400000,2026-03-11,10,19000.14,2026-01-01 19:11:00,call,0.00
2,400000,2026-03-11,10,19000.14,2026-01-08 14:37:00,payment,6772.92
3,400000,2026-03-11,10,19000.14,2026-01-10 08:04:00,call,0.00
4,400000,2026-03-11,10,19000.14,2026-01-12 14:06:00,payment,8731.15


In [ ]:
snapshot_actions.shape

(44972, 7)

### Фильтрация по датам

In [ ]:
snapshot_actions = snapshot_actions[snapshot_actions['action_dttm'] < snapshot_actions['snapshot_date']]

In [ ]:
(snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date']).sum()

np.int64(0)

Остаются только события с action_dttm < snapshot_date, что предотвращает утечку информации из будущего (data leakage).

### Расчёт признаков по окнам 7 и 30 дней

In [ ]:
window_calls_7d = snapshot_actions[(snapshot_actions['action_type'] == 'call') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=7))]

calls_last_7d = (window_calls_7d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(calls_last_7d=('action_type', 'count')))

In [ ]:
window_sms_7d = snapshot_actions[(snapshot_actions['action_type'] == 'sms') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=7))]

sms_last_7d = (window_sms_7d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(sms_last_7d=('action_type', 'count')))

In [ ]:
window_promises_30d = snapshot_actions[(snapshot_actions['action_type'] == 'promise') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=30))]

promises_agg = (window_promises_30d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(promises_last_30d=('action_type', 'count'),
         promised_amount_last_30d=('action_amount', 'sum')))

In [ ]:
window_payments_30d = snapshot_actions[(snapshot_actions['action_type'] == 'payment') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=30))]

payments_amount_last_30d = (window_payments_30d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(payments_amount_last_30d=('action_amount', 'sum')))

In [ ]:
payments_all = snapshot_actions[snapshot_actions['action_type'] == 'payment']

days_since_last_payment = (payments_all
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(last_payment_date=('action_dttm', 'max')))

days_since_last_payment['days_since_last_payment'] = (days_since_last_payment['snapshot_date'] - days_since_last_payment['last_payment_date']).dt.days
days_since_last_payment = days_since_last_payment.drop(columns='last_payment_date')

### Итоговая витрина

In [ ]:
feature_mart = snapshot.copy()

feature_mart = feature_mart.merge(calls_last_7d, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(sms_last_7d, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(promises_agg, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(payments_amount_last_30d, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(days_since_last_payment[['customer_id', 'snapshot_date', 'days_since_last_payment']], on=['customer_id', 'snapshot_date'], how='left')

count_sum_cols = ['calls_last_7d',
                  'sms_last_7d',
                  'promises_last_30d',
                  'payments_amount_last_30d',
                  'promised_amount_last_30d']

feature_mart[count_sum_cols] = feature_mart[count_sum_cols].fillna(0)

In [ ]:
feature_mart.head()

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,promises_last_30d,promised_amount_last_30d,payments_amount_last_30d,days_since_last_payment,contacts_last_7d
0,400000,2026-03-11,10,19000.14,0.0,0.0,0.0,0.00,0.00,57.0,0.0
1,400000,2026-03-15,22,16914.99,0.0,0.0,0.0,0.00,0.00,61.0,0.0
2,400001,2026-03-30,26,16142.60,0.0,1.0,1.0,10794.71,21487.22,0.0,1.0
3,400001,2026-04-03,33,16326.84,0.0,0.0,1.0,10794.71,21487.22,4.0,0.0
4,400002,2026-02-21,90,44636.91,1.0,0.0,0.0,0.00,0.00,40.0,1.0


In [ ]:
feature_mart.tail()

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,promises_last_30d,promised_amount_last_30d,payments_amount_last_30d,days_since_last_payment,contacts_last_7d
4495,401498,2026-03-08,13,15018.98,0.0,0.0,0.0,0.00,0.00,NaN,0.0
4496,401498,2026-04-06,25,13529.55,1.0,0.0,1.0,10304.35,11592.19,22.0,1.0
4497,401499,2026-02-19,48,48450.98,1.0,0.0,0.0,0.00,6316.16,0.0,1.0
4498,401499,2026-02-24,53,44179.69,0.0,1.0,0.0,0.00,6316.16,5.0,1.0
4499,401499,2026-04-07,53,41331.81,0.0,1.0,0.0,0.00,0.00,47.0,1.0


In [ ]:
feature_mart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               4500 non-null   int64         
 1   snapshot_date             4500 non-null   datetime64[ns]
 2   dpd                       4500 non-null   int64         
 3   outstanding_amount        4500 non-null   float64       
 4   calls_last_7d             4500 non-null   float64       
 5   sms_last_7d               4500 non-null   float64       
 6   promises_last_30d         4500 non-null   float64       
 7   promised_amount_last_30d  4500 non-null   float64       
 8   payments_amount_last_30d  4500 non-null   float64       
 9   days_since_last_payment   3061 non-null   float64       
 10  contacts_last_7d          4500 non-null   float64       
dtypes: datetime64[ns](1), float64(8), int64(2)
memory usage: 386.8 KB


### Итоговая схема витрины

| column_name | description |
|-------------|-------------|
| customer_id | идентификатор клиента |
| snapshot_date | дата среза |
| dpd | просрочка на дату среза |
| outstanding_amount | сумма задолженности на дату среза |
| calls_last_7d | количество звонков за последние 7 дней до snapshot_date |
| sms_last_7d | количество SMS за последние 7 дней до snapshot_date |
| promises_last_30d | количество обещаний за последние 30 дней до snapshot_date |
| payments_amount_last_30d | сумма платежей за последние 30 дней до snapshot_date |
| days_since_last_payment | число дней с момента последнего платежа до snapshot_date |
| promised_amount_last_30d | сумма обещаний за последние 30 дней до snapshot_date |

- Для count/sum-признаков отсутствие событий интерпретируется как 0.
- Для days_since_last_payment отсутствие исторических платежей оставляется как NaN.

### Top-20 клиентов

In [ ]:
feature_mart['contacts_last_7d'] = (feature_mart['calls_last_7d'] + feature_mart['sms_last_7d'])

top_clients_base = feature_mart[(feature_mart['payments_amount_last_30d'] == 0) & (feature_mart['contacts_last_7d'] > 0)].copy()

idx = top_clients_base.groupby('customer_id')['contacts_last_7d'].idxmax()

top_20_clients = (top_clients_base.loc[idx,['customer_id',
                                            'snapshot_date',
                                            'dpd',
                                            'outstanding_amount',
                                            'calls_last_7d',
                                            'sms_last_7d',
                                            'contacts_last_7d',
                                            'payments_amount_last_30d']]
                  .sort_values('contacts_last_7d', ascending=False).head(20).reset_index(drop=True))

In [ ]:
top_20_clients

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,contacts_last_7d,payments_amount_last_30d
0,400784,2026-02-02,72,15362.99,1.0,4.0,5.0,0.0
1,400470,2026-02-14,51,31271.78,1.0,3.0,4.0,0.0
2,401296,2026-04-24,103,30408.20,2.0,2.0,4.0,0.0
3,400820,2026-04-09,85,35028.88,1.0,3.0,4.0,0.0
4,401164,2026-04-09,41,49944.68,2.0,2.0,4.0,0.0
5,400729,2026-04-12,114,34103.46,2.0,2.0,4.0,0.0
6,401486,2026-03-23,67,10598.69,3.0,1.0,4.0,0.0
7,400985,2026-02-09,33,78428.12,4.0,0.0,4.0,0.0
8,401423,2026-02-19,59,27350.83,2.0,1.0,3.0,0.0
9,400708,2026-02-17,50,49058.48,2.0,1.0,3.0,0.0


# БЫСТРЫЕ ОТВЕТЫ:

## Python-1. Очистка данных и базовая витрина по взысканию

In [ ]:
def preprocess_data(cases: pd.DataFrame, contacts: pd.DataFrame, payments: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
  """Приводит даты к datetime, чистит дубли в contacts и убирает неположительные платежи."""
  cases['start_date'] = pd.to_datetime(cases['start_date'])
  contacts['event_dttm'] = pd.to_datetime(contacts['event_dttm'])
  payments['payment_dttm'] = pd.to_datetime(payments['payment_dttm'])

  contacts = contacts.sort_values('event_dttm').drop_duplicates(subset='event_id', keep='last')
  payments = payments[payments['payment_amount'] > 0]

  return cases, contacts, payments

In [ ]:
def filter_by_window(df: pd.DataFrame, event_col: str, start_col: str, days: int = 6) -> pd.DataFrame:
  """Выделяет события в окне [start_date, start_date + days]."""
  return df[(df[event_col] >= df[start_col]) & (df[event_col] <= df[start_col] + pd.Timedelta(days=days))]

In [ ]:
def make_collection_summary(cases: pd.DataFrame, contacts: pd.DataFrame, payments: pd.DataFrame, days: int = 6) -> pd.DataFrame:
  """Считает итоговую витрину по взысканию."""

  # total_cases
  summary = (cases
             .groupby('dpd_bucket', as_index=False)
             .agg(total_cases=('case_id', 'nunique'))
             .copy())

  # contacted_cases_7d
  contacts_cases = contacts.merge(
      cases[['case_id', 'start_date', 'dpd_bucket']],
      on='case_id',
      how='inner')

  contacts_window = filter_by_window(
      df=contacts_cases,
      event_col='event_dttm',
      start_col='start_date',
      days=days)

  contacted_cases = (contacts_window[['dpd_bucket', 'case_id']]
      .drop_duplicates()
      .groupby('dpd_bucket', as_index=False)
      .agg(contacted_cases_7d=('case_id', 'nunique')))

  # paid_cases_7d
  payments_cases = payments.merge(
      cases[['case_id', 'account_id', 'start_date', 'dpd_bucket']],
      on='account_id',
      how='inner')

  payments_window = filter_by_window(
      df=payments_cases,
      event_col='payment_dttm',
      start_col='start_date',
      days=days)

  paid_cases = (payments_window[['dpd_bucket', 'case_id']]
      .drop_duplicates()
      .groupby('dpd_bucket', as_index=False)
      .agg(paid_cases_7d=('case_id', 'nunique')))

  # collected_amount_7d
  collected_amount = (payments_window
      .groupby('dpd_bucket', as_index=False)
      .agg(collected_amount_7d=('payment_amount', 'sum')))

  # merge фичей в summary
  summary = summary.merge(contacted_cases, on='dpd_bucket', how='left')
  summary = summary.merge(paid_cases, on='dpd_bucket', how='left')
  summary = summary.merge(collected_amount, on='dpd_bucket', how='left')

  # заполняем пропуски
  metric_cols = ['contacted_cases_7d', 'paid_cases_7d', 'collected_amount_7d']
  summary[metric_cols] = summary[metric_cols].fillna(0)

  # avg_payment_per_paid_case_7d
  summary['avg_payment_per_paid_case_7d'] = (summary['collected_amount_7d'] / summary['paid_cases_7d'])
  summary['avg_payment_per_paid_case_7d'] = (summary['avg_payment_per_paid_case_7d'].replace([np.inf, -np.inf], 0).fillna(0))

  # сортировка бакетов
  bucket_order = ['1-30', '31-60', '61-90', '90+']
  summary['dpd_bucket'] = pd.Categorical(
      summary['dpd_bucket'],
      categories=bucket_order,
      ordered=True)
  summary = summary.sort_values('dpd_bucket').reset_index(drop=True)

  return summary

In [ ]:
def build_collection_summary(cases: pd.DataFrame, contacts: pd.DataFrame, payments: pd.DataFrame) -> pd.DataFrame:
  """Функция построения витрины."""
  cases = cases.copy()
  contacts = contacts.copy()
  payments = payments.copy()

  # --- проверки пустоты (базовые входные данные)
  if cases.empty:
      raise ValueError("cases is empty")

  if contacts.empty:
      print("Warning: contacts is empty")

  if payments.empty:
      print("Warning: payments is empty")

  cases, contacts, payments = preprocess_data(cases, contacts, payments)
  if payments.empty:
      print("Warning: no valid payments after filtering")

  summary = make_collection_summary(cases, contacts, payments, days=6)

  return summary

In [ ]:
summary = build_collection_summary(cases, contacts, payments)

In [ ]:
summary

,dpd_bucket,total_cases,contacted_cases_7d,paid_cases_7d,collected_amount_7d,avg_payment_per_paid_case_7d
0,1-30,1086,492,323,3367920.36,10426.998019
1,31-60,755,419,170,2349000.61,13817.650647
2,61-90,489,340,90,1317981.54,14644.239333
3,90+,270,196,20,345482.94,17274.147000


## Python-2. Анализ A/B-теста по SMS-напоминаниям

- Полный код в отдельных пунктах

Таблицы:

In [ ]:
segment_metrics

,segment,group_name,customers_cnt,delivered_rate,payment_rate_7d,avg_paid_amount_per_customer_7d
0,high_risk,control,863,0.900348,0.123986,1234.474762
1,high_risk,treatment,854,0.879391,0.148712,1424.542541
2,low_risk,control,987,0.959473,0.313070,1763.592665
3,low_risk,treatment,987,0.954407,0.372847,2083.512199
4,medium_risk,control,1296,0.932099,0.221451,1623.122955
5,medium_risk,treatment,1324,0.939577,0.237915,1767.628595
6,newcomers,control,454,0.916300,0.180617,1191.874802
7,newcomers,treatment,435,0.935632,0.204598,1296.637839


In [ ]:
overall_metrics

,group_name,customers_cnt,delivered_rate,payment_rate_7d,avg_paid_amount_per_customer_7d
0,control,3600,0.930000,0.218056,1514.082275
1,treatment,3600,0.928889,0.249722,1715.934331


In [ ]:
uplift_table = overall_metrics.set_index('group_name')[['payment_rate_7d']].T

uplift_table['abs_uplift'] = (uplift_table['treatment'] - uplift_table['control'])
uplift_table['rel_uplift'] = (uplift_table['abs_uplift'] / uplift_table['control'])

uplift_table

group_name,control,treatment,abs_uplift,rel_uplift
payment_rate_7d,0.218056,0.249722,0.031667,0.145223


In [ ]:
tests_table = pd.DataFrame({'test_name': ['z-test', 'chi-square'],
                            'statistic': [z_stat, chi2_stat],
                            'p_value': [p_value_z, p_value_chi]})

tests_table['is_significant_0.05'] = tests_table['p_value'] < 0.05

tests_table

,test_name,statistic,p_value,is_significant_0.05
0,z-test,3.173861,0.001504,True
1,chi-square,9.897444,0.001655,True


Вывод:

Эффект является не только статистически значимым, но и практически заметным:
- В группе `treatment` наблюдается **более высокий** `payment_rate_7d` по сравнению с `control` — **(+3.17 п.п.)**, что указывает на **положительный эффект нового текста SMS**.
- Результаты статистических тестов (`z-test` и `chi-test`) показывают, что **различие является статистически значимым** (`p-value < 0.05`), что позволяет отвергнуть нулевую гипотезу об отсутствии эффекта.
- Таким образом, можно сделать вывод, что **новый текст SMS действительно увеличивает вероятность оплаты клиентов в течение 7 дней**.
- **Дополнительно** наблюдается рост среднего платежа на клиента, что усиливает практическую значимость результата.

## Python-3. Построение дневной витрины признаков

- Полный код в отдельных пунктах

Итоговая витрина

In [ ]:
feature_mart.head()

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,promises_last_30d,promised_amount_last_30d,payments_amount_last_30d,days_since_last_payment,contacts_last_7d
0,400000,2026-03-11,10,19000.14,0.0,0.0,0.0,0.00,0.00,57.0,0.0
1,400000,2026-03-15,22,16914.99,0.0,0.0,0.0,0.00,0.00,61.0,0.0
2,400001,2026-03-30,26,16142.60,0.0,1.0,1.0,10794.71,21487.22,0.0,1.0
3,400001,2026-04-03,33,16326.84,0.0,0.0,1.0,10794.71,21487.22,4.0,0.0
4,400002,2026-02-21,90,44636.91,1.0,0.0,0.0,0.00,0.00,40.0,1.0


In [ ]:
feature_mart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               4500 non-null   int64         
 1   snapshot_date             4500 non-null   datetime64[ns]
 2   dpd                       4500 non-null   int64         
 3   outstanding_amount        4500 non-null   float64       
 4   calls_last_7d             4500 non-null   float64       
 5   sms_last_7d               4500 non-null   float64       
 6   promises_last_30d         4500 non-null   float64       
 7   promised_amount_last_30d  4500 non-null   float64       
 8   payments_amount_last_30d  4500 non-null   float64       
 9   days_since_last_payment   3061 non-null   float64       
 10  contacts_last_7d          4500 non-null   float64       
dtypes: datetime64[ns](1), float64(8), int64(2)
memory usage: 386.8 KB


Схема витрины

| column_name | description |
|-------------|-------------|
| customer_id | идентификатор клиента |
| snapshot_date | дата среза |
| dpd | просрочка на дату среза |
| outstanding_amount | сумма задолженности на дату среза |
| calls_last_7d | количество звонков за последние 7 дней до snapshot_date |
| sms_last_7d | количество SMS за последние 7 дней до snapshot_date |
| promises_last_30d | количество обещаний за последние 30 дней до snapshot_date |
| payments_amount_last_30d | сумма платежей за последние 30 дней до snapshot_date |
| days_since_last_payment | число дней с момента последнего платежа до snapshot_date |
| promised_amount_last_30d | сумма обещаний за последние 30 дней до snapshot_date |

- Для count/sum-признаков отсутствие событий интерпретируется как 0.
- Для days_since_last_payment отсутствие исторических платежей оставляется как NaN.

Top-20

In [ ]:
top_20_clients

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,contacts_last_7d,payments_amount_last_30d
0,400784,2026-02-02,72,15362.99,1.0,4.0,5.0,0.0
1,400470,2026-02-14,51,31271.78,1.0,3.0,4.0,0.0
2,401296,2026-04-24,103,30408.20,2.0,2.0,4.0,0.0
3,400820,2026-04-09,85,35028.88,1.0,3.0,4.0,0.0
4,401164,2026-04-09,41,49944.68,2.0,2.0,4.0,0.0
5,400729,2026-04-12,114,34103.46,2.0,2.0,4.0,0.0
6,401486,2026-03-23,67,10598.69,3.0,1.0,4.0,0.0
7,400985,2026-02-09,33,78428.12,4.0,0.0,4.0,0.0
8,401423,2026-02-19,59,27350.83,2.0,1.0,3.0,0.0
9,400708,2026-02-17,50,49058.48,2.0,1.0,3.0,0.0
